# Principal Component Analysis (PCA)

PCA is one of the most important dimensionality reduction techniques in machine learning. It reduces the number of features while retaining most of the information in the dataset.

## 1. Introduction to PCA

PCA finds the principal components (directions of maximum variance) in the data and projects the data onto these components.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%matplotlib inline

## 2. PCA Implementation and Visualization

Apply PCA to a 2D dataset and visualize the principal components.

In [ ]:
from sklearn.datasets import make_blobs

# Generate a 2D dataset
np.random.seed(0)
mean = [0, 0]
cov = [[6, 4], [4, 4]]
x = np.random.multivariate_normal(mean, cov, 200)

plt.figure(figsize=(8, 6))
plt.scatter(x[:, 0], x[:, 1], alpha=0.5)
plt.axis('equal')
plt.title('Original 2D Data')
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

In [ ]:
# YOUR CODE HERE - Fit PCA to the 2D data and visualize principal components

pca = PCA(n_components=2)
pca.fit(x)

print('Principal Components (eigenvectors):')
print(pca.components_)
print('\nExplained Variance:')
print(pca.explained_variance_)
print('\nExplained Variance Ratio:')
print(pca.explained_variance_ratio_)

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(x[:, 0], x[:, 1], alpha=0.5)

# Plot principal component arrows
origin = pca.mean_
for i, (comp, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    plt.annotate('', xy=(origin[0] + comp[0] * var, origin[1] + comp[1] * var),
                 xytext=(origin[0], origin[1]),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))

plt.axis('equal')
plt.title('PCA - Principal Components')
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

In [ ]:
# YOUR CODE HERE - Reduce to 1D and reconstruct

pca_1d = PCA(n_components=1)
x_reduced = pca_1d.fit_transform(x)
x_reconstructed = pca_1d.inverse_transform(x_reduced)

plt.figure(figsize=(8, 6))
plt.scatter(x[:, 0], x[:, 1], alpha=0.3, label='Original')
plt.scatter(x_reconstructed[:, 0], x_reconstructed[:, 1], alpha=0.5, color='red', label='Reconstructed (1 PC)')
plt.axis('equal')
plt.legend()
plt.title('PCA Reduction to 1D and Reconstruction')
plt.show()

print(f'Variance retained: {pca_1d.explained_variance_ratio_[0]:.4f}')

## 3. PCA with Facial Images

Apply PCA to the Labeled Faces in the Wild (LFW) dataset to understand how PCA can compress facial images.

In [ ]:
from sklearn.datasets import fetch_lfw_people

# Load the LFW dataset
faces = fetch_lfw_people(min_faces_per_person=60)
print(f'Image shape: {faces.images[0].shape}')
print(f'Number of images: {faces.images.shape[0]}')
print(f'Number of features: {faces.data.shape[1]}')
print(f'Number of people: {len(faces.target_names)}')

In [ ]:
# Display some sample faces
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(faces.images[i], cmap='gray')
    ax.set_title(faces.target_names[faces.target[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE - Apply PCA to facial images and show eigenfaces

pca_faces = PCA(n_components=150)
pca_faces.fit(faces.data)

# Show eigenfaces (principal components)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(pca_faces.components_[i].reshape(faces.images[0].shape), cmap='gray')
    ax.set_title(f'PC {i+1}')
    ax.axis('off')
plt.suptitle('Top 10 Eigenfaces (Principal Components)')
plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE - Reconstruct faces with different number of components

fig, axes = plt.subplots(4, 5, figsize=(12, 10))
n_components_list = [10, 50, 100, 150]

for row, n_comp in enumerate(n_components_list):
    pca_temp = PCA(n_components=n_comp)
    x_pca = pca_temp.fit_transform(faces.data)
    x_inv = pca_temp.inverse_transform(x_pca)
    for col in range(5):
        axes[row, col].imshow(x_inv[col].reshape(faces.images[0].shape), cmap='gray')
        if col == 0:
            axes[row, col].set_ylabel(f'n={n_comp}')
        axes[row, col].axis('off')

plt.suptitle('Face Reconstruction with Different Number of Components')
plt.tight_layout()
plt.show()

## 4. Choosing the Number of Components - Scree Plot

Use explained variance ratio and scree plots to determine the optimal number of components.

In [ ]:
# YOUR CODE HERE - Create scree plot for the facial images dataset

pca_full = PCA().fit(faces.data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
axes[0].plot(pca_full.explained_variance_ratio_[:50], 'bo-')
axes[0].set_xlabel('Component Number')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')

# Cumulative explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(cumulative_variance, 'ro-')
axes[1].axhline(y=0.90, color='k', linestyle='--', label='90% variance')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

# Find number of components for 90% and 95% variance
n_90 = np.argmax(cumulative_variance >= 0.90) + 1
n_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f'Components for 90% variance: {n_90}')
print(f'Components for 95% variance: {n_95}')

In [ ]:
# YOUR CODE HERE - Use PCA with explained variance threshold

pca_auto = PCA(n_components=0.95)
x_auto = pca_auto.fit_transform(faces.data)
print(f'Number of components for 95% variance: {pca_auto.n_components_}')
print(f'Original shape: {faces.data.shape}')
print(f'Reduced shape: {x_auto.shape}')

## 5. Filtering Noise with PCA

PCA can be used to filter noise from data by projecting to lower dimensions and reconstructing.

In [ ]:
from sklearn.datasets import load_digits

# Load digits dataset
digits = load_digits()
print(f'Shape: {digits.data.shape}')
print(f'Image shape: {digits.images[0].shape}')

# Display some digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Original Digits')
plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE - Add noise to digits and use PCA to filter it

np.random.seed(0)
noisy_digits = digits.data + np.random.normal(0, 4, digits.data.shape)

# Show noisy digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(noisy_digits[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Noisy Digits')
plt.tight_layout()
plt.show()

In [ ]:
# YOUR CODE HERE - Use PCA to denoise the digits

pca_denoise = PCA(n_components=0.50)
x_denoised = pca_denoise.fit_transform(noisy_digits)
x_reconstructed = pca_denoise.inverse_transform(x_denoised)

print(f'Components used: {pca_denoise.n_components_}')

# Show denoised digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_reconstructed[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Denoised Digits (PCA)')
plt.tight_layout()
plt.show()

## 6. Data Anonymization using PCA

PCA can be used to transform sensitive data into a lower-dimensional space, effectively anonymizing it while retaining useful patterns.

In [ ]:
# YOUR CODE HERE - Demonstrate data anonymization with PCA

# Create sample sensitive data
np.random.seed(42)
n_samples = 100
data = pd.DataFrame({
    'Age': np.random.randint(20, 70, n_samples),
    'Income': np.random.randint(30000, 150000, n_samples),
    'Credit_Score': np.random.randint(300, 850, n_samples),
    'Debt': np.random.randint(0, 50000, n_samples),
    'Years_Employed': np.random.randint(0, 40, n_samples)
})

print('Original Data (first 5 rows):')
print(data.head())

# Standardize
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Apply PCA to anonymize
pca_anon = PCA(n_components=3)
data_anonymized = pca_anon.fit_transform(data_scaled)

print(f'\nAnonymized Data Shape: {data_anonymized.shape}')
print(f'Variance retained: {sum(pca_anon.explained_variance_ratio_):.4f}')
print('\nAnonymized Data (first 5 rows):')
print(pd.DataFrame(data_anonymized[:5], columns=['PC1', 'PC2', 'PC3']))

## 7. High-dimensional Data Visualization

Use PCA to reduce high-dimensional data to 2D or 3D for visualization and cluster detection.

In [ ]:
# YOUR CODE HERE - Visualize high-dimensional data using PCA

digits = load_digits()

# Reduce to 2D
pca_2d = PCA(n_components=2)
digits_2d = pca_2d.fit_transform(digits.data)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(digits_2d[:, 0], digits_2d[:, 1], c=digits.target, 
                      cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, label='Digit')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Digits Dataset - PCA 2D Visualization')
plt.show()

print(f'Variance explained by 2 components: {sum(pca_2d.explained_variance_ratio_):.4f}')

In [ ]:
# YOUR CODE HERE - 3D visualization

from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3)
digits_3d = pca_3d.fit_transform(digits.data)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(digits_3d[:, 0], digits_3d[:, 1], digits_3d[:, 2], 
                     c=digits.target, cmap='tab10', alpha=0.5, s=10)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.title('Digits Dataset - PCA 3D Visualization')
plt.colorbar(scatter, label='Digit')
plt.show()

print(f'Variance explained by 3 components: {sum(pca_3d.explained_variance_ratio_):.4f}')